# Reading the Problem: The Step Before Writing Any Code

The most common intro-programming mistake isn't a syntax error -- it's
starting to type before actually reading the problem. This notebook makes
that step EXPLICIT and inspectable: for each problem, GIVENS / FIND /
CONSTRAINTS are extracted and printed BEFORE any code runs. Four examples,
deliberately mixing CS and physics problems, to show the same methodology
works regardless of subject. Engine: `dgs/reading_the_problem.py`.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

from dgs.reading_the_problem import (
    TWO_SUM, PROJECTILE_RANGE, TEMPS_ABOVE_FREEZING, OHMS_LAW_SAFETY_FILTER,
    print_reading_breakdown,
)
print("Setup complete.")


Setup complete.


## Example 1 (CS): Two Sum

Read the raw text FIRST. Notice it asks for INDICES, not the two numbers
themselves -- a detail easy to miss if you skim straight to coding.


In [2]:
print(TWO_SUM.raw_text)


Given a list of integers `nums` and an integer `target`, return the indices of the two numbers that add up to `target`. Assume exactly one solution exists, and you may not use the same element twice.


In [3]:
print("GIVENS:     ", TWO_SUM.givens)
print("FIND:       ", TWO_SUM.find)
print("CONSTRAINTS:", TWO_SUM.constraints)


GIVENS:      {'nums': [2, 7, 11, 15], 'target': 9}
FIND:        the pair of INDICES (not values) whose nums sum to target
CONSTRAINTS: ['exactly one solution exists', 'cannot reuse the same element twice']


In [4]:
result = TWO_SUM.run()
print("RESULT:", result)
nums = TWO_SUM.givens['nums']
print(f"Check: nums[{result[0]}] + nums[{result[1]}] = {nums[result[0]]} + {nums[result[1]]} = {nums[result[0]]+nums[result[1]]}")


RESULT: (0, 1)
Check: nums[0] + nums[1] = 2 + 7 = 9


## Example 2 (Physics): Projectile Range

A physics word problem has the SAME structure: given values, a target
quantity, and implicit assumptions (flat ground, no air resistance) that
are easy to gloss over but change the formula if violated.


In [5]:
print(PROJECTILE_RANGE.raw_text)
print()
print("GIVENS:     ", PROJECTILE_RANGE.givens)
print("FIND:       ", PROJECTILE_RANGE.find)
print("CONSTRAINTS:", PROJECTILE_RANGE.constraints)
print()
print("RESULT:", PROJECTILE_RANGE.run(), "meters")


A ball is launched at 20 m/s at an angle of 30 degrees above horizontal, on flat ground, with g=9.8 m/s^2. Find the horizontal range (how far it travels before landing).

GIVENS:      {'v0_ms': 20.0, 'angle_deg': 30.0, 'g_ms2': 9.8}
FIND:        the horizontal range R (a single number, in meters)
CONSTRAINTS: ['flat ground (launch height = landing height)', 'no air resistance']

RESULT: 35.34797566467096 meters


## Example 3 (Mixed CS + Physics): Temperatures Above Freezing

A unit-conversion trap hides in this one: "above freezing (0 Celsius)"
means the F->C conversion happens FIRST, and "above" is strict -- exactly
32F (0C) does NOT count. Reading the problem catches this before it
becomes an off-by-one-style bug.


In [6]:
print(TEMPS_ABOVE_FREEZING.raw_text)
print()
print("GIVENS:     ", TEMPS_ABOVE_FREEZING.givens)
print("FIND:       ", TEMPS_ABOVE_FREEZING.find)
print("CONSTRAINTS:", TEMPS_ABOVE_FREEZING.constraints)
print()
temps = TEMPS_ABOVE_FREEZING.givens['temps_f']
celsius = [(t-32)*5/9 for t in temps]
for f, c in zip(temps, celsius):
    print(f"  {f:5.1f} F -> {c:6.2f} C  {'(above freezing)' if c>0 else '(NOT above freezing)'}")
print()
print("RESULT:", TEMPS_ABOVE_FREEZING.run())


Given a list of daily temperatures in Fahrenheit, count how many days were above freezing (0 Celsius).

GIVENS:      {'temps_f': [28.0, 33.0, 32.0, 45.0, 10.0, 50.5]}
FIND:        a COUNT (single integer), not the converted list itself
CONSTRAINTS: ['freezing is 0C, i.e. exactly 32F -- strictly ABOVE, not at or above']

   28.0 F ->  -2.22 C  (NOT above freezing)
   33.0 F ->   0.56 C  (above freezing)
   32.0 F ->   0.00 C  (NOT above freezing)
   45.0 F ->   7.22 C  (above freezing)
   10.0 F -> -12.22 C  (NOT above freezing)
   50.5 F ->  10.28 C  (above freezing)

RESULT: 3


## Example 4 (Mixed CS + Physics): Ohm's Law Safety Filter

Same trap as example 3, different domain: "exceed the safe current limit"
is strict. One of the five readings sits EXACTLY at the limit
(5V / 10ohm = 0.5A) -- reading the problem is what tells you that reading
should NOT be flagged.


In [7]:
print(OHMS_LAW_SAFETY_FILTER.raw_text)
print()
print("GIVENS:     ", OHMS_LAW_SAFETY_FILTER.givens)
print("FIND:       ", OHMS_LAW_SAFETY_FILTER.find)
print("CONSTRAINTS:", OHMS_LAW_SAFETY_FILTER.constraints)
print()
V = OHMS_LAW_SAFETY_FILTER.givens['voltages_V']
R = OHMS_LAW_SAFETY_FILTER.givens['resistance_ohm']
limit = OHMS_LAW_SAFETY_FILTER.givens['max_current_A']
for i, v in enumerate(V):
    I = v/R
    flag = "EXCEEDS limit" if I > limit else ("at the limit exactly -- NOT flagged" if I == limit else "safe")
    print(f"  index {i}: V={v}V -> I={I:.2f}A  ({flag})")
print()
print("RESULT (flagged indices):", OHMS_LAW_SAFETY_FILTER.run())


A fixed resistor of 10 ohms is tested at several supply voltages. Given the list of voltages and a maximum safe current of 0.5 A, find the INDICES of the voltage readings that would exceed the safe current limit (V=IR).

GIVENS:      {'voltages_V': [2.0, 4.0, 5.0, 6.0, 3.0], 'resistance_ohm': 10.0, 'max_current_A': 0.5}
FIND:        the INDICES (not the voltages or currents themselves) that violate the limit
CONSTRAINTS: ['resistance is fixed across all readings', 'strictly EXCEEDS the limit, not equals it']

  index 0: V=2.0V -> I=0.20A  (safe)
  index 1: V=4.0V -> I=0.40A  (safe)
  index 2: V=5.0V -> I=0.50A  (at the limit exactly -- NOT flagged)
  index 3: V=6.0V -> I=0.60A  (EXCEEDS limit)
  index 4: V=3.0V -> I=0.30A  (safe)

RESULT (flagged indices): [3]


## Why This Matters

Every one of these four problems has a detail that only shows up if you
actually read the CONSTRAINTS before coding: Two Sum wants indices, not
values; the projectile problem assumes flat ground; the temperature
problem's "above" is strict; the Ohm's law problem's "exceed" is strict.
None of these are hard to fix once noticed -- but if you start writing
code from the FIND alone, skipping the constraints, each one becomes a
bug you discover later instead of a detail you handled up front.

This is the same discipline this repo's own research notebooks follow at
a much larger scale: every one of them states a theory section, derives
it, and VERIFIES the result against something independent before trusting
it -- "read the problem" is that same habit, at the very first step.
